In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader import data as pdr

In [ ]:
## Pulling ticker data for later use ## 
portfolio_groups = {
    "Dom_Eq": ['SPY', 'QQQ', 'IWM'],
    "Intl_Eq": ['EFA', 'EEM'],
    "Fixed_Inc": ['AGG', 'TLT', 'LQD'],
    "Alt": ['GLD', 'VNQ'],
    "Factor": ['MTUM', 'VLUE', 'QUAL', 'USMV']
}
all_tickers = [ticker for group in portfolio_groups.values() for ticker in group]
portfolio_data : pd.DataFrame = yf.download(all_tickers, # type: ignore
                                            period = "5y",
                                            auto_adjust= False,
                                            group_by="column")
if portfolio_data is None:
    raise ValueError("Download failed")
adj_close = portfolio_data["Adj Close"] 
daily_returns = adj_close.pct_change().dropna()


[*********************100%***********************]  14 of 14 completed


Ticker,AGG,EEM,EFA,GLD,IWM,LQD,MTUM,QQQ,QUAL,SPY,TLT,USMV,VLUE,VNQ
Date,,,,,,,,,,,,,,
2021-07-09,-0.003274,0.017287,0.017001,0.003321,0.021110,-0.004210,0.020547,0.006244,0.008800,0.010675,-0.014063,0.006983,0.017312,0.016447
2021-07-12,-0.000605,0.000934,0.004273,-0.001241,0.000840,-0.000519,0.010819,0.003906,0.003031,0.003582,-0.001297,0.000267,0.004589,0.008566
2021-07-13,-0.002162,0.000933,-0.005256,0.001302,-0.018846,-0.003711,-0.010302,0.000000,-0.001843,-0.003409,-0.007653,-0.002134,-0.009707,-0.013967
2021-07-14,0.003381,0.004287,0.002265,0.010755,-0.015430,0.004618,-0.007287,0.001793,0.000960,0.001492,0.011362,0.003608,-0.000288,0.006987
2021-07-15,0.002160,0.002413,-0.009540,0.000175,-0.005666,0.002150,-0.002330,-0.007024,-0.003762,-0.003416,0.011031,0.000932,-0.004518,0.001616
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-30,-0.003925,0.014534,0.004157,-0.000543,0.004950,-0.005743,0.020327,0.017015,0.009849,0.007787,-0.011778,0.003120,-0.005277,-0.017524
2026-07-01,-0.001510,-0.028212,-0.008279,0.006026,-0.003761,-0.002098,-0.042966,-0.015250,-0.002552,-0.001353,-0.006759,0.004458,-0.021721,0.004044
2026-07-02,0.001117,-0.011733,0.013104,0.020318,-0.005813,0.001660,-0.035264,-0.017334,-0.002010,-0.001314,-0.000117,0.010321,-0.012790,0.012394


In [ ]:
start = daily_returns.index.min()
end = daily_returns.index.max()
rfr = pdr.DataReader("DGS3MO", "fred", start=start, end = end)
rfr['risk_free_rate'] = rfr['DGS3MO'] / 100
rfr['daily_rfr'] = rfr['risk_free_rate'] / 252
daily_rfr = rfr['daily_rfr'].reindex(daily_returns.index).ffill()


In [ ]:
rfr

,DGS3MO,risk_free_rate,daily_rfr
DATE,,,
2021-07-09,0.06,0.0006,0.000002
2021-07-12,0.05,0.0005,0.000002
2021-07-13,0.05,0.0005,0.000002
2021-07-14,0.06,0.0006,0.000002
2021-07-15,0.05,0.0005,0.000002
...,...,...,...
2026-06-30,3.87,0.0387,0.000154
2026-07-01,3.85,0.0385,0.000153
2026-07-02,3.82,0.0382,0.000152


In [ ]:
## Pulling Metadata for later use ##
ticker_metadata = {
    ticker:group
    for group, tickers in portfolio_groups.items()
    for ticker in tickers
}
ticker_meta_df = (
    pd.DataFrame.from_dict(ticker_metadata, orient = "index", columns = ['group'])
    .reset_index()
    .rename(columns = {"index":"ticker"})
)

In [ ]:
beta_series = pd.Series(index = daily_returns.columns, dtype = float)

benchmark_var = daily_returns['SPY'].var()

for col in daily_returns.columns:
    cov = daily_returns[col].cov(daily_returns['SPY'])
    beta = cov / benchmark_var
    beta_series[col] = beta

beta_series

Ticker
AGG     0.074158
EEM     0.771452
EFA     0.774277
GLD     0.154005
IWM     1.115562
LQD     0.186606
MTUM    1.074670
QQQ     1.261131
QUAL    0.988952
SPY     1.000000
TLT     0.068937
USMV    0.595370
VLUE    0.907897
VNQ     0.731636
dtype: float64

In [ ]:
## Feature Engineering ## 
daily_mean_returns = daily_returns.mean()
annual_expected_returns = daily_returns.mean() * 252
average_trade_volume = portfolio_data['Volume'].mean()
annual_volatility = daily_returns.std() * np.sqrt(252)
corr_matrix = daily_returns.corr()
cov_matrix = daily_returns.cov() * 252
rolling_vol = daily_returns.rolling(30).std() * np.sqrt(252)
cumulative_returns = (1 + daily_returns).cumprod()
running_max = cumulative_returns.cummax()
drawdowns = (cumulative_returns - running_max) / running_max
portfolio_metrics = {
    "returns": annual_expected_returns,
    "volatility": annual_volatility,
    "beta": beta_series,
    "correlation": corr_matrix,
    "covariance": cov_matrix,
    "drawdowns": drawdowns
}

In [ ]:
metrics_table = pd.DataFrame()
metrics_table['expected_returns'] = daily_mean_returns.round(6)
metrics_table['ann_returns'] = annual_expected_returns
metrics_table['annual_volatility'] = annual_volatility
metrics_table['beta'] = beta_series
metrics_table['av_trade_volume'] = average_trade_volume
metrics_table['ann_sharpe'] = (
    daily_returns.sub(daily_rfr, axis=0).mean()*252) / (daily_returns.std()*(252**0.5))
metrics_table['max_drawdowns'] = drawdowns.min()


In [ ]:
portfolio_data.to_csv("data/processed/portfolio_data.csv")
daily_returns.to_csv("data/processed/daily_returns.csv")
adj_close.to_csv("data/processed/adj_close.csv")
cov_matrix.to_csv("data/processed/cov_matrix.csv")
metrics_table.to_csv("data/processed/metrics_table.csv")
daily_rfr.to_csv("data/processed/daily_risk_free_rate.csv")